# Task 4: Open-Set Recognition (CIFAR) — GPU Colab (clone workflow)

Use this when the **code Drive account is out of GPU quota** (same pattern as Task 3).

1. Sign into Colab with the **GPU account** (account B).
2. Runtime → GPU.
3. This notebook **clones GitHub** (does **not** need account-A Drive for code).
4. CIFAR-10/100 download via torchvision into `/content` (no Task-2 checkpoint needed).
5. After runs, download / copy `task4_results_bundle.zip` back into account-A `task4/results/`.

**Hard rules:** checkpoint = CIFAR-10 val Acc only; thresholds from CIFAR-10 val only; no CIFAR-100 in train / ckpt / score design / thresholds.

In [ ]:
import torch

print("cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU — Runtime → Change runtime type → GPU, then re-run.")
print("device:", torch.cuda.get_device_name(0))

## Clone repo from GitHub

Public clone into `/content/ATML-PA1` (fast local disk). Optional: mount **this** account's Drive only to **save the results zip** at the end — not required for training.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ttqureshi/ATML-PA1.git"
REPO_DIR = Path("/content/ATML-PA1")

if (REPO_DIR / ".git").is_dir():
    print("Repo exists — pulling latest main...")
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "main"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"])
else:
    print("Cloning", REPO_URL)
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("cwd:", Path.cwd())
print("task4:", (REPO_DIR / "task4").is_dir())

In [ ]:
%pip install -q -r requirements.txt

## Stage 1 — CIFAR-10 stratified splits (seed 6304)

Downloads CIFAR-10 if needed. Writes `task4/results/splits/cifar10_seed6304.json`.

In [ ]:
!python -m task4.scripts.run_task4 --stages splits

## Stage 2 — Train Vanilla + GCSC + PROSER

~100 + 100 + 50 epochs on GPU. Checkpoint = CIFAR-10 **val Acc** only.

In [ ]:
!python -m task4.scripts.run_task4 --stages train_main

## Stage 3 — Extract logits / features

Caches identical inputs for every post-hoc score.

In [ ]:
!python -m task4.scripts.run_task4 --stages extract_all

## Stage 4 — OSR evaluation

Tables + score-distribution figure + Vanilla MLS failure examples.
Thresholds from CIFAR-10 **val** only.

In [ ]:
!python -m task4.scripts.run_task4 --stages eval

## Bundle results → bring back to account A

1. Download `/content/task4_results_bundle.zip`, **or**
2. Uncomment Drive mount below and copy to **this** (GPU) account's Drive, then transfer to account A.

On account A, unzip into `task4/results/` (merge tables/curves/figures/cache/checkpoints/splits).

In [ ]:
from pathlib import Path
import shutil

src = Path("task4/results")
zip_base = Path("/content/task4_results_bundle")
zip_path = zip_base.with_suffix(".zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_base), "zip", root_dir=src)
print("Wrote", zip_path, "bytes:", zip_path.stat().st_size)

# Optional: save on GPU account Drive, then move file to account-A Drive
# from google.colab import drive
# drive.mount("/content/drive")
# out = Path("/content/drive/MyDrive/task4_results_bundle.zip")
# shutil.copy(zip_path, out)
# print("Copied to", out)

from google.colab import files
files.download(str(zip_path))